# IdiomX Dataset Construction Notebook

**Author:** Ayman Ali Sharara  
**Project:** IdiomX – Neural Understanding of English Idioms  
**github** : https://github.com/aymanshar/idiomx-dataset
**Year:** 2026  

---

## Description
This notebook documents the full pipeline for constructing the IdiomX dataset,
including data collection, filtering, normalization and merging.

### Dataset Flow in IdiomX
The enrichment pipeline operates on the following dataset stages:

    IdiomX_dataset/
    │    
    ├── scripts/
    │   ├── extract_idioms_from_kaikki.py
    │   ├── collect_02_filter_strict_idioms.py
    │   ├── collect_03_clean_idioms.py
    │   ├── collect_04_build_high_precision_idioms.py
    │   ├── collect_05_normalize_kaikki_high_precision.py
    │   ├── collect_09_extract_wordnet_multiword_expressions.py
    │   ├── collect_10_merge_wordnet_with_kaikki.py
    │   ├── collect_12_01_filter_global_idioms.py
    │   └── finalize_pre_enrichment_dataset.py
    │
    ├── notebooks/
    │   └── 01_data_collection.ipynb
    │
    ├── data/
    │   ├── raw/
    │   |   └── kaikki.org-dictionary-English-words.jsonl
    │   ├── enriched/
    │   ├── processed/
    │   └── final/
    │       ├── idiomx_core.csv
    │       ├── dataset_statistics.json
    │       ├── idiomx_core.parquet
    │       ├── idiomx_human_examples_only.parquet
    │       └── idiomx_human_examples_only.csv
    │
    ├── config/
    │   ├── api_config.py
    │   └── prompts.py
    ├── docs/
    │   └── IdiomX_Dataset_Paper_v2.pdf
    └── README.md

---

## License
This work is released under the MIT License.

---

## Citation
If you use this dataset, please cite the IdiomX research paper.

## Run the pipeline using python files (CMD)
from anaconda CMD

go to root_folder/data_collection/scripts 

```bash
conda create -n idiomx python=3.11 -y
source $(conda info --base)/etc/profile.d/conda.sh
conda activate idiomx

pip install -r scripts/requirements.txt
```

run the python files in the same order
```bash
python extract_idioms_from_kaikki.py
python collect_02_filter_strict_idioms.py
python collect_03_clean_idioms.py
python collect_04_build_high_precision_idioms.py
python collect_05_normalize_kaikki_high_precision.py
python collect_09_extract_wordnet_multiword_expressions.py
python collect_10_merge_wordnet_with_stage3.py
python collect_12_01_filter_global_idioms.py
python finalize_pre_enrichment_dataset.py.py
```

In [1]:
from pathlib import Path
import sys

# Project directories
BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "data"
DATA_RAW_DIR = DATA_DIR / "raw"
DATA_PROCESS_DIR = DATA_DIR / "processed"
DATA_FINAL_DIR = DATA_DIR / "final"
DATA_SAMPLE_DIR = DATA_DIR / "sample"

# Raw dataset file
KAIKKI_FILE = DATA_RAW_DIR / "kaikki.org-dictionary-English-words.jsonl"

# External dataset sources if required
KAIKKI_DATASET_URL = "https://kaikki.org/dictionary/English/words/kaikki.org-dictionary-English-words.jsonl"

# python file path 
SCRIPTS_DIR = BASE_DIR / "scripts"

# Make the scripts folder importable
sys.path.append(str(SCRIPTS_DIR))

# Make sure directories exist
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESS_DIR.mkdir(parents=True, exist_ok=True)
DATA_FINAL_DIR.mkdir(parents=True, exist_ok=True)
DATA_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# download the kaikki.org-dictionary-English-words
import requests

#file_path = KAIKKI_FILE

if not KAIKKI_FILE.exists():
    print("Downloading Kaikki dataset...")
    # External dataset sources
    url = KAIKKI_DATASET_URL
    
    r = requests.get(url, stream=True)

    with open(KAIKKI_FILE, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    print("Download complete")
else:
    print("Dataset already exists")

Dataset already exists


## 1. Extract Idioms from Wiktionary (Kaikki Dataset)

Define the input and output paths used in the idiom extraction pipeline.  
The raw Kaikki Wiktionary dataset is specified as the primary input source.  

In [4]:
#=======================
# this part define the main rote incase if started the notebook from here for this step
#=======================
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
DATA_PROCESS_DIR = DATA_DIR / "processed"
#======================================

# output extracted files (Extract Idioms from Wiktionary)
KAIKKI_CSV = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki.csv"
KAIKKI_JSONL = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki.jsonl"

In [7]:
# import the function from scripts/ extraction python file
from extract_idioms_from_kaikki import extract_kaikki_idioms

# run extraction
summary = extract_kaikki_idioms(
    input_file=KAIKKI_FILE,
    output_csv=KAIKKI_CSV,
    output_jsonl=KAIKKI_JSONL,
    strict_mode=False,
)

#summary

Reading Kaikki JSONL: 1454988it [02:24, 10077.69it/s]


Extraction finished.
{'input_file': '..\\data\\raw\\kaikki.org-dictionary-English-words.jsonl', 'output_csv': '..\\data\\processed\\idioms_wiktionary_kaikki.csv', 'output_jsonl': '..\\data\\processed\\idioms_wiktionary_kaikki.jsonl', 'strict_mode': False, 'total_lines_scanned': 1454988, 'rows_kept': 229807}


{'input_file': '..\\data\\raw\\kaikki.org-dictionary-English-words.jsonl',
 'output_csv': '..\\data\\processed\\idioms_wiktionary_kaikki.csv',
 'output_jsonl': '..\\data\\processed\\idioms_wiktionary_kaikki.jsonl',
 'strict_mode': False,
 'total_lines_scanned': 1454988,
 'rows_kept': 229807}

In [5]:
import pandas as pd

df_kaikki = pd.read_csv(KAIKKI_CSV, encoding="utf-8-sig")
print("dataset shape: ",df_kaikki.shape)
df_kaikki.head()

dataset shape:  (229807, 8)


,idiom,meaning,example,pos,tags,idiom_hint,source,sense_index
0,rain cats and dogs,To rain very heavily.,"[""]Those weather-gaws aren't out for nothing. ...",verb,"idiomatic, impersonal",1,kaikki_wiktionary,0
1,Pope Julius,A sixteenth-century gambling card game about w...,Of Pope Julius cardys he ys chefe cardynall.,name,obsolete,0,kaikki_wiktionary,0
2,GNU FDL,partial Initialism of GNU Free Documentation L...,NaN,name,NaN,0,kaikki_wiktionary,0
3,current events,Current affairs; those events and issues of in...,The teacher asked the students to write a repo...,noun,"plural, plural-normally",0,kaikki_wiktionary,0
4,false friend,A word in a language that bears a deceptive re...,A word and its false friend may well be etymol...,noun,NaN,0,kaikki_wiktionary,0


In [6]:
print("Rows:", len(df_kaikki))
print("Unique idioms:", df_kaikki["idiom"].nunique())
print("\nPOS distribution:")
print(df_kaikki["pos"].value_counts().head(10))

Rows: 229807
Unique idioms: 193390

POS distribution:
pos
noun           127869
verb            51656
name            33492
adj              4508
phrase           3486
prep_phrase      3484
adv              1947
intj             1612
proverb           943
prep              331
Name: count, dtype: int64


## 2. Strict Filtering of Extracted Wiktionary Idioms

The initial extraction from the Kaikki Wiktionary dataset includes many lexical entries that are not strictly idiomatic expressions.  
To improve dataset quality, a strict filtering stage is applied to retain only strong idiom candidates.

This filtering step removes entries that are likely to represent literal phrases, incomplete expressions, or non-idiomatic multi-word constructions. The filtered output produces a cleaner subset of idioms that can later be merged with other idiom datasets.

In [7]:
#=======================
# this part define the main rote incase if started the notebook from here for this step
#=======================
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
DATA_PROCESS_DIR = DATA_DIR / "processed"
#======================================

# input file
INPUT_FILE_kaikki = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki.csv"

# output file
OUTPUT_FILE_kaikki_strict = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_strict.csv"

In [8]:
# import the function from scripts/ filter_strict_idioms
from collect_02_filter_strict_idioms import filter_strict_idioms

# run filtering function
df_kaikki_strict = filter_strict_idioms(INPUT_FILE_kaikki,OUTPUT_FILE_kaikki_strict )

print(df_kaikki_strict.shape)
df_kaikki_strict.head()

Saved: ..\data\processed\idioms_wiktionary_kaikki_strict.csv
Rows: 30749
(30749, 8)


,idiom,meaning,example,pos,tags,idiom_hint,source,sense_index
0,$100 hamburger,A general aviation flight that involves flying...,It’s called hunting the $100 hamburger — “$100...,noun,slang,0,kaikki_wiktionary,0
1,$100 hamburger,Used other than figuratively or idiomatically:...,Daniel Boulud has a restaurant that serves a t...,noun,,1,kaikki_wiktionary,1
2,& ux.,Obsolete form of et ux..,,phrase,"alt-of, obsolete",0,kaikki_wiktionary,0
3,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,phrase,informal,0,kaikki_wiktionary,0
4,'ark at 'ee,Used to draw attention to something or someone.,Then a lady came into the shop and saw the T-s...,phrase,informal,0,kaikki_wiktionary,1


In [9]:
before_count = len(df_kaikki)
after_count = len(df_kaikki_strict)

print("Before strict filtering:", before_count)
print("After strict filtering:", after_count)
print("Retention rate:", round(after_count / before_count * 100, 2), "%")
print("Unique idioms:", df_kaikki_strict["idiom"].nunique())

df_kaikki_strict.sample(10, random_state=42)

Before strict filtering: 229807
After strict filtering: 30749
Retention rate: 13.38 %
Unique idioms: 24938


,idiom,meaning,example,pos,tags,idiom_hint,source,sense_index
19795,one nail drives out another,"New people, things, or customs eventually supp...",,proverb,,0,kaikki_wiktionary,0
21000,play away,To be sexually unfaithful out of one's home,,verb,"intransitive, slang",0,kaikki_wiktionary,0
10493,follow the crowd,"To conform to majority beliefs, opinions, or p...","As much as the next guy, I don't like to follo...",verb,idiomatic,1,kaikki_wiktionary,0
10919,fuck this for a game of soldiers,Synonym of sod this for a game of soldiers.,"Fuck this for a game of soldiers, I'm gonna ni...",phrase,vulgar,0,kaikki_wiktionary,0
7363,crap circus,A thoroughly chaotic and disorganized situatio...,I didn't have a scanner with me but it certain...,noun,"slang, vulgar",0,kaikki_wiktionary,0
9434,emperor's new clothes,Something obvious and embarrassing that is pol...,Marks & Spencer announced a 19% fall in annual...,noun,"idiomatic, plural, plural-only",1,kaikki_wiktionary,0
15181,jail bars,Used other than figuratively or idiomatically:...,,noun,"plural, plural-only",1,kaikki_wiktionary,1
20202,pa chew cheng,Of a man: to masturbate.,"2001, Anonymous, Worst Job in Singapore, Googl...",verb,"Singapore, colloquial",0,kaikki_wiktionary,0
3298,be cruel to be kind,To act cruelly in order to achieve a positive ...,"I muſt be cruell only to be kinde, / This bad ...",verb,idiomatic,1,kaikki_wiktionary,0
2276,anal vore,To insert a character into the rectum.,"In the comic, the giant predator anal vores it...",verb,"informal, slang",0,kaikki_wiktionary,0


## 3. Cleaning the Strict Idiom Dataset

An additional cleaning stage is performed to normalize and refine the extracted idioms.

By standardizes text fields, removes residual noise, and ensures that idioms and their associated metadata follow a consistent format.

In [10]:
#=======================
# this part define the main rote incase if started the notebook from here for this step
#=======================
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
DATA_PROCESS_DIR = DATA_DIR / "processed"
#======================================

# input file
INPUT_FILE_kaikki_strict= DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_strict.csv"
#output file
OUTPUT_FILE_kaikki_cleaned = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_cleaned.csv"

In [11]:
# import the function from scripts/ clean_idioms
from collect_03_clean_idioms import clean_idioms

# run clean function
df_kaikki_clean = clean_idioms(INPUT_FILE_kaikki_strict,OUTPUT_FILE_kaikki_cleaned)

print(df_kaikki_clean.shape)
df_kaikki_clean.head()

Saved: ..\data\processed\idioms_wiktionary_kaikki_cleaned.csv
Rows: 21816
(21816, 8)


,idiom,meaning,example,pos,tags,idiom_hint,source,sense_index
0,$100 hamburger,A general aviation flight that involves flying...,It’s called hunting the $100 hamburger — “$100...,noun,slang,0,kaikki_wiktionary,0
1,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,phrase,informal,0,kaikki_wiktionary,0
2,'d best,Had best.,It's getting late. You'd best get on home.,verb,"auxiliary, colloquial, modal",0,kaikki_wiktionary,0
3,'er indoors,A person's wife.,,pron,"UK, slang",0,kaikki_wiktionary,0
4,'fraid so,I am afraid so,,phrase,slang,0,kaikki_wiktionary,0


## 4. Constructing a High-Precision Idiom Subset

Identify idioms with stronger evidence of figurative usage.  

By applying additional heuristics and linguistic indicators to prioritize expressions that are more likely to represent true idiomatic constructions. These include indicators such as idiom-related tags, semantic descriptions suggesting figurative meaning, and patterns commonly associated with idiomatic expressions.

In [12]:
#=======================
# this part define the main rote incase if started the notebook from here for this step
#=======================
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
DATA_PROCESS_DIR = DATA_DIR / "processed"
#======================================

INPUT_FILE_kaikki_cleaned = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_cleaned.csv"
OUTPUT_FILE_kaikki_high_precision = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_high_precision.csv"

In [13]:
# import the function from scripts/ build_high_precision_idioms
from collect_04_build_high_precision_idioms import high_precision_idioms

df_kaikki_high_precision = high_precision_idioms(INPUT_FILE_kaikki_cleaned,OUTPUT_FILE_kaikki_high_precision )

print(df_kaikki_high_precision.shape)
df_kaikki_high_precision.head()

Saved: ..\data\processed\idioms_wiktionary_kaikki_high_precision.csv
Rows: 13202
(13202, 8)


,idiom,meaning,example,pos,tags,idiom_hint,source,sense_index
0,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,phrase,informal,0,kaikki_wiktionary,0
1,'fraid so,I am afraid so,,phrase,slang,0,kaikki_wiktionary,0
2,'nuff said,"Used either to end a discussion, or to imply t...","“Your new computer is super expensive!” “Yeah,...",phrase,informal,0,kaikki_wiktionary,0
3,'tis the season,Indicating that it is the time of year around ...,"Anyway, ‛tis the season, and apropos of that a...",phrase,,0,kaikki_wiktionary,0
4,110 proof,Stronger than strong.,"1941 February 9, ""The 'Hated Hun'—Then and Now...",adj,"idiomatic, not-comparable",1,kaikki_wiktionary,0


## 5. Normalizing High-Precision Wiktionary Idioms

A normalization step is applied to standardize the structure and textual representation of the entries.

To ensures that idioms, meanings, examples, and metadata follow a consistent schema across all records. 

Text normalization includes:

    - Standardizing formatting, 
    - Removing irregular whitespace, 
    - Aligning column structures 
    
so the dataset can be easily merged with idioms collected from other sources.

In [14]:
#=======================
# this part define the main rote incase if started the notebook from here for this step
#=======================
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
DATA_PROCESS_DIR = DATA_DIR / "processed"
#======================================

INPUT_FILE_kaikki_high_precision = DATA_PROCESS_DIR / "idioms_wiktionary_kaikki_high_precision.csv"
OUTPUT_FILE_source_kaikki = DATA_PROCESS_DIR / "idioms_source_kaikki.csv"

In [15]:
# building high precision normalized idioms
from collect_05_normalize_kaikki_high_precision import normalize_high_precision_idioms

# run normalize_high_precision_idioms
df_kaikki_normalized = normalize_high_precision_idioms(INPUT_FILE_kaikki_high_precision,OUTPUT_FILE_source_kaikki)

print(df_kaikki_normalized.shape)
df_kaikki_normalized.head()

Saved: ..\data\processed\idioms_source_kaikki.csv
Rows: 13202
(13202, 9)


,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url
0,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,kaikki_wiktionary,dictionary,phrase,informal,high,
1,'fraid so,I am afraid so,,kaikki_wiktionary,dictionary,phrase,slang,high,
2,'nuff said,"Used either to end a discussion, or to imply t...","“Your new computer is super expensive!” “Yeah,...",kaikki_wiktionary,dictionary,phrase,informal,high,
3,'tis the season,Indicating that it is the time of year around ...,"Anyway, ‛tis the season, and apropos of that a...",kaikki_wiktionary,dictionary,phrase,,high,
4,110 proof,Stronger than strong.,"1941 February 9, ""The 'Hated Hun'—Then and Now...",kaikki_wiktionary,dictionary,adj,"idiomatic, not-comparable",high,


### Note: 
    Some intermediate data sources were removed due to licensing constraints.
    Numbering of scripts is preserved for reproducibility and traceability.

## 6. Extracting Multi-Word Expressions from WordNet

To expand the lexical coverage of idiomatic expressions, multi-word expressions are extracted from the WordNet lexical database. WordNet is a widely used lexical resource that organizes English words into synonym sets (synsets) and provides semantic relationships and definitions.

In [16]:
#=======================
# this part define the main rote incase if started the notebook from here for this step
#=======================
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
DATA_PROCESS_DIR = DATA_DIR / "processed"
#======================================

OUTPUT_FILE_wordnet = DATA_PROCESS_DIR / "idioms_source_wordnet.csv"

In [17]:
# extract_wordnet_multiword_expressions
from collect_09_extract_wordnet_multiword_expressions import extract_wordnet_multiword_expressions

df_wordnet = extract_wordnet_multiword_expressions(OUTPUT_FILE_wordnet)

print(df_wordnet.shape)
df_wordnet.head()

Saved: ..\data\processed\idioms_source_wordnet.csv
Rows: 68072
Unique idioms: 64188
(68072, 9)


,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url
0,ready to hand,easy to reach,,wordnet,lexical_database,s,,medium,
1,out of reach,inaccessibly located or situated,,wordnet,lexical_database,s,,medium,
2,dead on target,accurately placed or thrown,,wordnet,lexical_database,s,,medium,
3,wide of the mark,not on target,,wordnet,lexical_database,s,,medium,
4,used to,in the habit; ; ; - Henry David Thoreau,,wordnet,lexical_database,s,,medium,


### Dataset Expansion with WordNet

After integrating WordNet multi-word expressions, the dataset contains more than 68,000 candidate expressions from WordNet alone. While many of these expressions are not strictly idiomatic, they significantly expand the lexical coverage of the dataset and will later be refined through global idiom filtering.

## 7. Merging WordNet Expressions with the Main Idiom Dataset

WordNet expressions are labeled with a medium idiom confidence level because they include both idiomatic and compositional multi-word expressions.

The candidate multi-word expressions extracted from WordNet are merged with the existing idiom dataset already contains curated idioms from Wiktionary.

In [18]:
#=======================
# this part define the main rote incase if started the notebook from here for this step
#=======================
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
DATA_PROCESS_DIR = DATA_DIR / "processed"
#======================================

KAIKKI_CSV = DATA_PROCESS_DIR / "idioms_source_kaikki.csv"
WORDNET_CSV = DATA_PROCESS_DIR / "idioms_source_wordnet.csv"
MERGED_KAIKKI_WORDNET_CSV = DATA_PROCESS_DIR / "idioms_merged_kaikki_wordnet.csv"

In [22]:
# merge_wordnet_with_kaikki
from collect_10_merge_wordnet_with_kaikki import merge_kaikki_with_wordnet

df_merged_kaikki_wordnet = merge_kaikki_with_wordnet(
    kaikki_file = KAIKKI_CSV,
    wordnet_file = WORDNET_CSV,
    output_file = MERGED_KAIKKI_WORDNET_CSV
)

print(df_merged_kaikki_wordnet.shape)
df_merged_kaikki_wordnet.head()

Saved: ..\data\processed\idioms_merged_kaikki_wordnet.csv
Rows: 81274
Unique idioms: 75373

Source distribution:
source
wordnet              68072
kaikki_wiktionary    13202
Name: count, dtype: int64
(81274, 9)


,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url
0,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,kaikki_wiktionary,dictionary,phrase,informal,high,
1,'fraid so,I am afraid so,,kaikki_wiktionary,dictionary,phrase,slang,high,
2,'nuff said,"Used either to end a discussion, or to imply t...","“Your new computer is super expensive!” “Yeah,...",kaikki_wiktionary,dictionary,phrase,informal,high,
3,'tis the season,Indicating that it is the time of year around ...,"Anyway, ‛tis the season, and apropos of that a...",kaikki_wiktionary,dictionary,phrase,,high,
4,110 proof,Stronger than strong.,"1941 February 9, ""The 'Hated Hun'—Then and Now...",kaikki_wiktionary,dictionary,adj,"idiomatic, not-comparable",high,


### Dataset Expansion after WordNet Integration

After integrating WordNet multi-word expressions, the dataset expanded significantly to more than 80,000 expressions. 

the dataset at this stage contains both idiomatic and non-idiomatic phrases. 

These expressions will be refined in the subsequent global idiom filtering stage to produce the final high-quality idiom dataset.

data/processed/

- idioms_dataset_stage.csv ->#  merged raw sources

- idioms_dataset_stage_broad.csv ->#  large dataset For training an LLM or large model

- idioms_dataset_stage5_high_precision.csv ->#  clean dataset for research

### IdiomX-Core

High quality idioms

    idioms_dataset_stage_high_precision.csv

### IdiomX-Extended

Broader idiomatic expressions

    idioms_dataset_stage_broad.csv

In [27]:
try:
    from rdflib import Graph, Literal
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rdflib"])
    from rdflib import Graph, Literal
try:
    import requests
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests"])
    import requests

## 8. Global Idiom Filtering

Now dataset contains a large number of expressions that are not strictly idiomatic. In particular, WordNet contributes many compositional multi-word expressions that must be filtered to ensure dataset quality.

A global idiom filtering procedure is applied to the entire merged dataset. The filtering process significantly reduces the dataset size while improving idiom quality and reliability.

In [23]:
#=======================
# this part define the main rote incase if started the notebook from here for this step
#=======================
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
DATA_PROCESS_DIR = DATA_DIR / "processed"
#======================================

MERGED_KAIKKI_WORDNET_CSV = DATA_PROCESS_DIR / "idioms_merged_kaikki_wordnet.csv"
DATASET_BROAD_CSV = DATA_PROCESS_DIR / "idioms_dataset_broad.csv"
DATASET_HIGH_CSV = DATA_PROCESS_DIR / "idioms_dataset_high_precision.csv"

In [25]:
# filter_global_idioms
from collect_12_01_filter_global_idioms import filter_global_idioms

df_broad, df_high, stats = filter_global_idioms(
    input_file=MERGED_KAIKKI_WORDNET_CSV,
    output_broad=DATASET_BROAD_CSV,
    output_high=DATASET_HIGH_CSV
)

print(stats)
print(df_broad.shape, df_high.shape)

df_broad.head()
df_high.head()

C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\notebooks\..\scripts\collect_12_01_filter_global_idioms.py:367: DtypeWarning: Columns (2,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file, encoding="utf-8-sig")


Saved broad: ..\data\processed\idioms_dataset_broad.csv
Saved high : ..\data\processed\idioms_dataset_high_precision.csv
Rows after basic cleaning: 78598
Rows broad: 15344
Rows high precision: 12853
Unique idioms broad: 15084
Unique idioms high precision: 12847

Broad per source:
source
kaikki_wiktionary    12838
wordnet               2506
Name: count, dtype: int64

High per source:
source
kaikki_wiktionary    12838
wordnet                 15
Name: count, dtype: int64
{'rows_input': 78598, 'rows_broad': 15344, 'rows_high_precision': 12853, 'unique_idioms_input': 72897, 'unique_idioms_broad': 15084, 'unique_idioms_high_precision': 12847, 'broad_source_counts': {'kaikki_wiktionary': 12838, 'wordnet': 2506}, 'high_source_counts': {'kaikki_wiktionary': 12838, 'wordnet': 15}}
(15344, 9) (12853, 9)


,idiom,meaning_en,example,source,source_type,pos,tags,idiom_confidence,source_url
0,'ark at 'ee,Listen to you; listen to yourself; listen to it.,‘Look at that water! No wonder Duddle said he ...,kaikki_wiktionary,dictionary,phrase,informal,high,
1,'fraid so,I am afraid so,,kaikki_wiktionary,dictionary,phrase,slang,high,
2,'nuff said,"Used either to end a discussion, or to imply t...","“Your new computer is super expensive!” “Yeah,...",kaikki_wiktionary,dictionary,phrase,informal,high,
3,'tis the season,Indicating that it is the time of year around ...,"Anyway, ‛tis the season, and apropos of that a...",kaikki_wiktionary,dictionary,phrase,,high,
4,110 proof,Stronger than strong.,"1941 February 9, ""The 'Hated Hun'—Then and Now...",kaikki_wiktionary,dictionary,adj,"idiomatic, not-comparable",high,


### Dataset Quality Improvement after Global Filtering

The global idiom filtering stage reduces the dataset from more than 75,000 candidate expressions to approximately 13,000 high-precision idioms. This filtering process removes a large number of compositional phrases originating primarily from WordNet, significantly improving the linguistic quality of the dataset.

The resulting high-precision idiom dataset represents the final curated idiom collection used in the IdiomX project.

#### Broad Dataset (larger coverage)

idioms_dataset_stage_broad.csv

Contains:

- idioms

- idiom-like expressions

- some multi-word expressions

- lexical phrases from WordNet

- looser filtering

This dataset is bigger but slightly noisier.

Example types included:

    break the ice
    kick the bucket
    take into account
    state of the art
    in the long run

##### Good for:

- training large language models

- semantic search

- embedding training

- exploratory analysis

#### High-Precision Dataset (very clean)

idioms_dataset_stage_high_precision.csv

Contains only strong idioms, based on:

- idiom tags

- phrase dictionaries

- idiom confidence

- high-quality sources

Example:

    break the ice
    kick the bucket
    spill the beans
    bite the bullet

#### Good for:

- academic dataset release

- idiom translation model

- idiom detection model

- benchmark dataset

## 9. Final Dataset Statistics Summary

The IdiomX dataset representing **12,853 unique idiomatic expressions** collected from multiple lexical resources.It provides a large-scale resource for studying idiomatic language, semantic interpretation, and idiom translation tasks.

In [31]:
#=======================
# this part define the main rote incase if started the notebook from here for this step
#=======================
from pathlib import Path
import sys

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
DATA_PROCESS_DIR = DATA_DIR / "processed"
DATA_SAMPLE_DIR = DATA_DIR / "sample"
SCRIPTS_DIR = BASE_DIR / "scripts"

sys.path.append(str(SCRIPTS_DIR))
#======================================

from finalize_pre_enrichment_dataset import finalize_pre_enrichment_dataset

stats, df_source_stats, df_pre_enrichment, df_sample = finalize_pre_enrichment_dataset(
    input_file=DATA_PROCESS_DIR / "idioms_dataset_high_precision.csv",
    stats_json=DATA_PROCESS_DIR / "idiomx_pre_enrichment_statistics.json",
    stats_csv=DATA_PROCESS_DIR / "idiomx_pre_enrichment_source_distribution.csv",
    output_csv=DATA_PROCESS_DIR / "idiomx_pre_enrichment.csv",
    output_parquet=DATA_PROCESS_DIR / "idiomx_pre_enrichment.parquet",
    sample_csv=DATA_SAMPLE_DIR / "idiomx_pre_enrichment_sample.csv",
    sample_n=500
)

print(stats)
display(df_source_stats.head())
display(df_pre_enrichment.head(3))
display(df_sample.head(3))

Saved JSON stats: ..\data\processed\idiomx_pre_enrichment_statistics.json
Saved source distribution CSV: ..\data\processed\idiomx_pre_enrichment_source_distribution.csv

Summary:
Rows total: 12853
Unique idioms: 12847
Rows with meaning: 12853
Rows with example: 8891
Average idiom tokens: 3.23

Source distribution:
source
kaikki_wiktionary    12838
wordnet                 15
Name: count, dtype: int64
Saved pre-enrichment files:
..\data\processed\idiomx_pre_enrichment.csv
..\data\processed\idiomx_pre_enrichment.parquet
..\data\processed\idiomx_pre_enrichment_statistics.json
..\data\processed\idiomx_pre_enrichment_source_distribution.csv
..\data\sample\idiomx_pre_enrichment_sample.csv

Pre-enrichment dataset shape: (12853, 15)
Sample dataset shape: (500, 15)
{'rows_total': 12853, 'unique_idioms': 12847, 'unique_meanings': 12525, 'rows_with_meaning': 12853, 'rows_with_example': 8891, 'rows_without_meaning': 0, 'rows_without_example': 3962, 'avg_idiom_tokens': 3.23, 'min_idiom_tokens': 2, '

,source,count
0,kaikki_wiktionary,12838
1,wordnet,15


,idiom_id,idiom_canonical,idiom_surface,example,idiom_canonical_meaning,source,source_type,pos,tags,idiom_confidence,source_url,record_origin,license_source,example_language,meaning_language
0,idiomx_543ff4605a0e,'ark at 'ee,'ark at 'ee,‘Look at that water! No wonder Duddle said he ...,Listen to you; listen to yourself; listen to it.,kaikki_wiktionary,dictionary,phrase,informal,high,NaN,source_only,wiktionary_cc_by_sa_4_0,en,en
1,idiomx_7fd1dd45c159,'fraid so,'fraid so,<NA>,I am afraid so,kaikki_wiktionary,dictionary,phrase,slang,high,NaN,source_only,wiktionary_cc_by_sa_4_0,en,en
2,idiomx_7312fd49eefe,'nuff said,'nuff said,"“Your new computer is super expensive!” “Yeah,...","Used either to end a discussion, or to imply t...",kaikki_wiktionary,dictionary,phrase,informal,high,NaN,source_only,wiktionary_cc_by_sa_4_0,en,en


,idiom_id,idiom_canonical,idiom_surface,example,idiom_canonical_meaning,source,source_type,pos,tags,idiom_confidence,source_url,record_origin,license_source,example_language,meaning_language
11751,idiomx_e3e7571a5cef,turkey shoot,turkey shoot,"In the Battle of the Marianas, which pilots ca...",A situation in which numerous weapons are disc...,kaikki_wiktionary,dictionary,noun,"broadly, idiomatic",high,NaN,source_only,wiktionary_cc_by_sa_4_0,en,en
8,idiomx_de44334e4d0c,51 percent,51 percent,"But in a political system, everything tends to...",A narrow or bare majority.,kaikki_wiktionary,dictionary,noun,"US, idiomatic",high,NaN,source_only,wiktionary_cc_by_sa_4_0,en,en
12518,idiomx_59881240777f,win by a nose,win by a nose,<NA>,To win by a small margin; to have a narrow vic...,kaikki_wiktionary,dictionary,verb,idiomatic,high,NaN,source_only,wiktionary_cc_by_sa_4_0,en,en


# Findings and Conclusions

## Dataset Construction Overview

In this notebook, we constructed a comprehensive English idiom dataset by aggregating multiple publicly available lexical resources. The collection pipeline involved extracting idiomatic expressions from several heterogeneous sources, normalizing the data schema, merging the datasets, and applying rigorous filtering and deduplication procedures to ensure dataset quality.

The sources used in the dataset include:

* Wiktionary (via the Kaikki dataset)
* WordNet multi-word expressions

* * *

## Source Contribution Analysis

### Broad Dataset

* Wiktionary (Kaikki): **15,410 idioms**
* WordNet: **2,506 expressions**


### High-Precision Dataset

* Wiktionary (Kaikki): **12,838 idioms**
* WordNet: **15 idioms**

This distribution shows that **Wiktionary contributes the majority of verified idioms**, while WordNet mainly contributes general multi-word expressions that are filtered out during the precision stage.

* * *

## Dataset Quality Assessment

The final **IdiomX v1 dataset** contains:

* **12,853 unique idiomatic expressions**
* **High definition coverage (~99.7%)**
* Multiple trusted lexical sources
* Structured metadata including:

    * idiom
    * English meaning
    * example sentence
    * source
    * part-of-speech tags
    * confidence level

Compared to many published idiom datasets—which typically contain between **5,000 and 10,000 idioms**—IdiomX v1 represents a **large and high-quality idiom resource suitable for NLP research**.

* * *

## Final Dataset Output

The final dataset is saved as:

```
    data/final/idiomx_dataset_v1.csv
```

Additional metadata files include:

```
    idiomx_dataset_v1_statistics.json
    idiomx_dataset_v1_source_distribution.csv
```

These files provide reproducibility and transparency regarding the dataset composition.

* * *

## Conclusion

This work produced **IdiomX v1**, a curated English idiom dataset built through a multi-source data integration pipeline. By combining dictionary resources, lexical databases, and idiom repositories, and by applying systematic filtering and deduplication, we obtained a high-quality dataset of more than **12,000 idiomatic expressions**.

The resulting dataset provides a strong foundation for several NLP tasks, including:

* idiom detection
* idiom interpretation
* machine translation of idioms
* figurative language modeling
* idiom-aware language models

In future work, the dataset will be extended with **multilingual translations**, beginning with English-Arabic idiom pairs, enabling the training of specialized translation models capable of handling figurative language.